# Class Activity: The Best Search Engine

A search engine can find every page that matches a query. The hard part of search is the order. A person reads the first five results and rarely looks further, so a search engine is judged on what it puts at the top.

In this activity you write one function, `score`, that gives a page a number for how well it answers a query. The search engine ranks the matching pages by that number. Everything else (the crawl, the index, PageRank) is done for you in `searchkit`.

Each submission is graded on twelve **hidden** queries, which are not the practice queries below. The leaderboard ranks the class by the result, highest first. Submit as often as you like during the session; the leaderboard keeps your best.

In [ ]:
import json
import searchkit

web = searchkit.Web()
practice = json.load(open("practice_judgments.json"))
print(len(practice), "practice queries:", ", ".join(practice))

## What `score` receives

For each query, the search engine finds every page that contains at least one of the query's words. It calls `score(features)` once for each of those pages. `features` is a dictionary:

| Key | Meaning |
|---|---|
| `pagerank` | The page's PageRank, from the lab. All pages together sum to 1. |
| `in_degree` | How many pages link to this page. |
| `title_matches` | How many of the query's words appear in the page's title. |
| `body_matches` | How many times the query's words appear in the page's text, counting repeats. |
| `all_words` | `True` if every word of the query appears somewhere on the page. |
| `query_words` | How many different words the query has. |
| `page_words` | How many words the page has. |

The features are numbers only. `score` does not see the URL or the query, so it has to judge a page by its features, as a search engine judges a page it has never seen.

Run the cell below to see the features of three pages for the query `flu vaccine`.

In [ ]:
for url in ["https://publichealth.example/topics/flu",
            "https://jhi.example/",
            "https://miraclecure.example/"]:
    print(web.features(url, "flu vaccine"))

## How a ranking is measured

Each practice query comes with **judgments**: for each page, 2 if it answers the query, 1 if it is related, and 0 for every page not listed. The measure is **nDCG@5** (normalized discounted cumulative gain over the top five). It rewards relevant pages near the top: a page at position 1 counts fully, and each position below counts a little less. A ranking that puts the best possible five pages in the best possible order scores 1. Missing a relevant page, or placing it low, costs points.

The cell below scores two simple strategies. The first ranks by PageRank alone. The second ranks by how often the query's words appear.

In [ ]:
def by_pagerank(f):
    return f["pagerank"]


def by_word_count(f):
    return f["body_matches"]


for name, strategy in [("PageRank alone", by_pagerank), ("word count alone", by_word_count)]:
    mean, per_query = searchkit.evaluate(web, strategy, practice)
    print(f"{name}: {100 * mean:.1f}")
    for query, value in per_query.items():
        print(f"    {query:20s} {100 * value:5.1f}")

Look at the top five for one query under each strategy. Neither strategy is good, and each fails in a different way.

In [ ]:
for url in web.search("flu vaccine", by_pagerank):
    print("PageRank:  ", url)
print()
for url in web.search("flu vaccine", by_word_count):
    print("word count:", url)

## Your `score` function

Edit `score` below. It must return a number, and it may use any of the features. Combine them however you like: add them, weight them, compare them, cap them.

Some questions to work from:

- Which pages in the top five above should not be there? What do their features have in common, and what do the correct pages have that they lack?
- The lecture showed that a link farm has many links pointing in, but each link carries little PageRank. Which two features together measure that?
- A page that repeats a word many times is not necessarily about that word. How could `page_words` help?

`score` is graded on queries you have not seen. A rule that fixes one practice query but makes no sense in general will not help on the leaderboard.

In [ ]:
def score(f):
    return 2 * f["title_matches"] + f["all_words"]

In [ ]:
mean, per_query = searchkit.evaluate(web, score, practice)
print(f"Your score on the practice queries: {100 * mean:.1f}")
for query, value in per_query.items():
    print(f"    {query:20s} {100 * value:5.1f}")

In [ ]:
query = "flu vaccine"
for url in web.search(query, score):
    print(url)

## Submitting

Submit the notebook. The grader runs your `score` on the twelve hidden queries and reports your nDCG@5, from 0 to 100, which places you on the leaderboard. Your best submission counts.

`score` must work from `features` alone. A function that reads files or remembers earlier calls is not a search engine, and it is removed from the leaderboard.